# 04 — Cohort Retention Analysis

**Business question:** for each month in which a customer first purchased, what percentage purchased again in each later calendar month?

This is monthly activity retention, not continuous survival.


In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().resolve()
while not (ROOT / 'src').exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import pandas as pd
from src.data_loader import load_olist_data

data = load_olist_data(ROOT / "data_raw")
customers = data["customers"]
orders = data["orders"]

customer_orders = orders.merge(
    customers[["customer_id", "customer_unique_id"]],
    on="customer_id",
    how="left"
)



## Step 1 — Assign each customer to their first-purchase month


In [ ]:
customer_orders["order_month"] = (
    customer_orders["order_purchase_timestamp"].dt.to_period("M")
)

customer_cohort = (
    customer_orders.groupby("customer_unique_id")["order_month"]
    .min()
    .rename("cohort_month")
)

customer_orders = customer_orders.merge(
    customer_cohort,
    on="customer_unique_id",
    how="left"
)



## Step 2 — Calculate the month index

For example, a January cohort has January = 0, February = 1, March = 2, and so on.


In [ ]:
customer_orders["cohort_index"] = (
    (customer_orders["order_month"].dt.year - customer_orders["cohort_month"].dt.year) * 12
    + (customer_orders["order_month"].dt.month - customer_orders["cohort_month"].dt.month)
)

cohort_counts = (
    customer_orders
    .groupby(["cohort_month", "cohort_index"])["customer_unique_id"]
    .nunique()
    .reset_index()
)

cohort_table = cohort_counts.pivot(
    index="cohort_month",
    columns="cohort_index",
    values="customer_unique_id"
)

cohort_sizes = cohort_table[0]
retention_matrix = (
    cohort_table.divide(cohort_sizes, axis=0) * 100
).round(2)

retention_clean = retention_matrix.loc["2017-01":"2018-08"]
retention_clean



## Step 3 — Average retention by cohort age


In [ ]:
average_retention = retention_clean.mean(axis=0).round(2)
average_retention.head(12)



Observed average monthly activity retention:

| Cohort index | Retention |
|---:|---:|
| Month 0 | 100.00% |
| Month 1 | 0.48% |
| Month 2 | 0.31% |
| Month 3 | 0.25% |
| Month 4 | 0.27% |
| Month 5 | 0.23% |
| Month 6 | 0.27% |
| Month 7 | 0.21% |
| Month 8 | 0.21% |
| Month 9 | 0.18% |
| Month 10 | 0.26% |
| Month 11 | 0.21% |

## Step 4 — Compare Month-1 retention by year

The August 2018 cohort is not fully comparable because its next month falls into sparse September data. Compare Jan-Jul 2018 with 2017 rather than interpreting the raw 2018 mean as a full-year result.


In [ ]:
month1_trend = (
    retention_clean[[1]]
    .rename(columns={1: "month_1_retention"})
    .reset_index()
)

month1_trend["year"] = month1_trend["cohort_month"].dt.year

valid_month1 = month1_trend[
    ~(
        (month1_trend["cohort_month"].dt.year == 2018)
        & (month1_trend["cohort_month"].dt.month == 8)
    )
]

valid_month1.groupby("year")["month_1_retention"].mean().round(2)



**Observed conclusion:** 2017 Month-1 retention averaged about 0.51%; comparable Jan-Jul 2018 averaged about 0.49%. Retention was broadly flat, not materially improving.

**Learning note:** later monthly activity can rise after an earlier decline because customers may skip a month and return. Do not interpret this matrix as a survival curve.
